#task 1
1. Geçen haftaki MLP + BatchNorm modelini videodaki gibi küçük adımlara böl (logits, counts, probs, logprobs, ...) ve loss.backward() ile her ara değişkenin gradient'ini al. (Egzersiz 1, videonun ilk yarısı)


In [14]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [15]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [16]:
#mapping
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [17]:
#dataset build
block_size =3
def build_dataset(words):
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y


import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr,  Ytr = build_dataset(words [:n1])     # 80%
Xdev, Ydev = build_dataset(words [n1:n2])  # 10%
Xte,  Yte  = build_dataset (words [n2:])   # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [18]:
def cmp(s,dt,t):
  ex = torch.all(dt==t.grad).item()
  app =torch.allclose(dt,t.grad)
  maxdiff = (dt-t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [19]:
n_embd = 10 # embedding vektor boyutu
n_hidden = 64 # MLPteki hidden layerdaki noron sayisi, Xvala gore degisebilir

g = torch.Generator().manual_seed(2147483647)
C = torch.randn(vocab_size,n_embd, generator=g)

#ilk layer
W1 = torch.randn((n_embd*block_size,n_hidden),generator=g) * (5/3)/((n_embd*block_size)**0.5) # kaiming init
b1 = torch.randn(n_hidden,generator=g) * 0.1 # bnbias oldugu icin suan ise yaramaz

# ikinci layer
W2 = torch.randn((n_hidden,vocab_size),generator=g) * 0.1
b2 = torch.randn(vocab_size,generator =g) *0.1

# batch norm parametreleri
bngain = torch.randn(1,n_hidden)*0.1 + 1.0 # her noron icin ayri parametre atiyoruz
bnbias= torch.randn(1,n_hidden)*0.1

parameters = [C,W1,b1,W2,b2,bngain,bnbias]
print(sum(p.nelement() for p in parameters)) # toplam parametre sayisi

for p in parameters:
  p.requires_grad = True


4137


In [20]:
batch_size = 32
n = batch_size

ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator=g)
Xb,Yb = Xtr[ix],Ytr[ix] # batch X,Y

In [21]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time
emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bi
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()
# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3270, grad_fn=<NegBackward0>)

torch.Size([32])

#task 2
2. Aynı gradient'leri elle yaz, cmp fonksiyonuyla tek tek karşılaştır. Hepsi "exact" ya da "approximate" olana kadar devam et. Takıldığın türevi videoda bul, ama önce kendin dene.

In [43]:
dh.shape

torch.Size([32, 64])

In [40]:
W2.shape

torch.Size([64, 27])

In [39]:
b2.shape

torch.Size([27])

In [42]:
logits.shape

torch.Size([32, 27])

In [22]:
#exercise 1

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n),Yb] = -1.0/n
cmp('logprobs',dlogprobs,logprobs) # loss = -logprobs(range(n),Yb).mean()

dprobs = (1.0/probs) * dlogprobs # dprobs/loss = dlogprobs/loss * dprobs/logprobs microgradde yaptigimiz out grad geliyor
# NOT!!!! pytorch log alma islemeini e tabaninda yaptigi icin lokal tureve 1/probs yazabildik yoksa 1/(probs*ln(taban)) olurdu
cmp('probs',dprobs,probs)


dcounts_sum_inv = (counts * dprobs).sum(1,keepdim=True) #Burada aynı counts_sum_inv loss'a 27 ayrı yoldan etki ediyor. Bir değişken birden fazla yerde kullanılıyorsa gradient'ler toplanır (micrograd'daki += bu
# anlama geliyor)
cmp('counts_sum_inv',dcounts_sum_inv,counts_sum_inv)

dcounts = (counts_sum_inv * dprobs) # dcounts_sum/loss = d_counts_sum/probs * dprobs/loss
# burada broadcasting duzgun calisiyor ama 2 farkli branch dcountsu etkiledigi icin simdi compare etmiyoruz


dcounts_sum = (-1/(counts_sum)**2) * dcounts_sum_inv
cmp('counts_sum',dcounts_sum,counts_sum)


dcounts += torch.ones_like(counts) * dcounts_sum # toplama isleminin turevi 1 oldugu icin router gorevi goruyor
# turevi yayiyor yani
cmp('counts',dcounts,counts)

dnorm_logits = norm_logits.exp() * dcounts
cmp('dnorm_logits',dnorm_logits,norm_logits)

dlogits = 1 * dnorm_logits
#cmp('logits',dlogits,logits) 2. branchi de var

dlogit_maxes = (-1 * dnorm_logits).sum(1,keepdim=True)
cmp('logit_maxes',dlogit_maxes,logit_maxes)

dlogits += F.one_hot(logits.max(1).indices,num_classes=logits.shape[1]) * dlogit_maxes
cmp('logits',dlogits,logits)


dh = dlogits @ W2.T # elle derive edilebilir fakat shape uzerinden mantik yurutmek cok daha kolay
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
cmp('W2',dW2,W2)
cmp('b2',db2,b2)
cmp('h',dh,h)


dhpreact = (1.0-h**2) * dh
cmp('hpreact',dhpreact,hpreact)

dbngain = (bnraw*dhpreact).sum(0,keepdim=True) # yine boyut matematigi
dbnbias = 1*dhpreact.sum(0,keepdim=True) # yine broadcasting
cmp('bngain',dbngain,bngain)
cmp('bnbias',dbnbias,bnbias)

dbnraw = bngain * dhpreact # dimensionlar uydugu icin sum etmeye gerek yok
cmp('bnraw',dbnraw,bnraw)

dbndiff = bnvar_inv * dbnraw
# cmp('bndiff',dbndiff,bndiff) # 2 li branch oldugu icin ekleme yapacagiz

dbnvar_inv = (bndiff*dbnraw).sum(0,keepdim=True) # dimension muhabbeti yine
cmp('bnvar_inv',dbnvar_inv,bnvar_inv)

dbnvar = -0.5 * (bnvar+1e-5)**-1.5 * dbnvar_inv
cmp('bnvar',dbnvar,bnvar)

dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
cmp('bndiff2',dbndiff2,bndiff2)

dbndiff+= (2*bndiff) * dbndiff2
cmp('bndiff',dbndiff,bndiff)


dhprebn = 1*dbndiff
#cmp('hprebn',dhprebn,hprebn) 1 branch daha var

dbnmeani = -1*dhprebn.sum(0)
cmp('bnmeani',dbnmeani,bnmeani)

dhprebn+= 1/n * dbnmeani
cmp('hprebn',dhprebn,hprebn)

dembcat = dhprebn @ W1.T
cmp('embcat',dembcat,embcat)

dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
cmp('W1',dW1,W1)
cmp('b1',db1,b1)



demb = dembcat.view(emb.shape)
cmp('emb',demb,emb) # burada sadece 32,30 luk matrisi tekrardan geri 32,3,10 haline cevirdik yani backprop
#burada "undo islemi yapmis oldu"

dc = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dc[ix] += demb[k,j]
cmp('C',dc,C)





logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
dnorm_logits    | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10
bngain          | exact: False | approximate: True  | maxdiff: 1.862645149230957e-09
bnbias          | exact: False | approximate: True  | maxdiff: 3.725290298461914e-09
bnraw   

In [23]:
# c11 c12 c13 = a11 a12 a13   b1
# c21 c22 c23 = a21 a22 a23 - b2
# C31 c32 с33 = а31 а32 а33   b3
# s0 e.g. c32 = a32 - b3 #yine broadcasting oldugu icin sum yapmamiz lazim

In [24]:
# not exact cikmamaisin sebebi pytorch surumu ve float32 bit hesabi